# Tutorial 6: Atlas-Free 3D CNN Models

This short tutorial compares the packaged MLP and atlas-free CNN on two familiar network maps. It reconstructs both maps, compares MLP and CNN contrastive similarities, and measures inference time. All models are loaded through `neurovlm.models.load_model`; no `experiments/` imports or path manipulation are needed.

The first run downloads the requested pretrained checkpoints from the NeuroVLM Hugging Face repositories.

In [ ]:
import time

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F
from nilearn.image import resample_to_img

from neurovlm.cnn import atlas_free_volume_to_mlp_flat, mlp_flat_to_atlas_free_volume
from neurovlm.data import load_dataset, load_masker
from neurovlm.models import load_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Prepare two example maps

The MLP consumes 28,542 masked voxels. The CNN consumes the equivalent cropped `(1, 36, 45, 38)` volume. The conversion below is a boolean scatter on the same 4 mm MNI grid, with no interpolation between the two model inputs.

In [ ]:
examples = [
    ("AUD", "auditory network"),
    ("DN-A", "default mode network"),
]
networks = load_dataset("networks")["Du"]
masker = load_masker()

flat_rows = []
for network_id, _ in examples:
    resource = networks[network_id]
    image = nib.Nifti1Image(resource["array"].astype(np.float32), resource["affine"])
    image_4mm = resample_to_img(
        image, masker.mask_img_, interpolation="nearest", force_resample=True, copy_header=True
    )
    flat_rows.append(torch.as_tensor(np.asarray(masker.transform(image_4mm)).reshape(-1)))

mlp_inputs = (torch.stack(flat_rows) > 0).float()
cnn_inputs = mlp_flat_to_atlas_free_volume(mlp_inputs)
assert torch.equal(atlas_free_volume_to_mlp_flat(cnn_inputs), mlp_inputs)
mlp_inputs.shape, cnn_inputs.shape

## Load MLP and CNN models

`autoencoder_cnn` selects the mixed-source CNN autoencoder. Domain-specific autoencoders are also available as `autoencoder_cnn_pubmed`, `autoencoder_cnn_nilearn`, and `autoencoder_cnn_neurovault`.

In [ ]:
mlp_autoencoder = load_model("autoencoder").to(device).eval()
cnn_autoencoder = load_model("autoencoder_cnn").to(device).eval()
cnn_contrastive = load_model("contrastive_cnn_pubmed").to(device).eval()
mlp_brain_projection = load_model("proj_head_image_infonce").to(device).eval()
mlp_text_projection = load_model("proj_head_text_infonce").to(device).eval()

## Compare reconstructions

Each row shows the same target map, the CNN reconstruction, and the MLP reconstruction scattered back into CNN volume space.

In [ ]:
with torch.inference_mode():
    cnn_recon = cnn_autoencoder(cnn_inputs.to(device)).clamp(0, 1).cpu()
    mlp_recon_flat = torch.sigmoid(mlp_autoencoder(mlp_inputs.to(device))).cpu()
    mlp_recon = mlp_flat_to_atlas_free_volume(mlp_recon_flat)

fig, axes = plt.subplots(len(examples), 3, figsize=(10, 6))
column_titles = ["Target", "CNN reconstruction", "MLP reconstruction"]
for row, ((network_id, _), target, cnn_pred, mlp_pred) in enumerate(zip(examples, cnn_inputs, cnn_recon, mlp_recon)):
    for column, volume in enumerate((target, cnn_pred, mlp_pred)):
        axes[row, column].imshow(volume[0, volume.shape[1] // 2].T, cmap="magma", origin="lower")
        axes[row, column].set_axis_off()
        if row == 0:
            axes[row, column].set_title(column_titles[column])
    axes[row, 0].set_ylabel(network_id)
fig.tight_layout()

## Compare contrastive matches

The CNN Stage 3 model expects the normalized, empty-string-centered SPECTER2 convention used during training. Diagonal cells should be stronger when each brain map matches its description.

In [ ]:
specter = load_model("specter").to(device)
descriptions = [description for _, description in examples]
with torch.inference_mode():
    text_raw = specter(descriptions)
    empty_center = specter([""])[0]
    text_embeddings = F.normalize(text_raw - empty_center, dim=-1)

    cnn_brain = cnn_contrastive.encode_brain(cnn_inputs.to(device))
    cnn_text = cnn_contrastive.encode_text(text_embeddings.to(device))
    cnn_similarity = (cnn_brain @ cnn_text.T).cpu()

    mlp_latent = mlp_autoencoder.encoder(mlp_inputs.to(device))
    mlp_brain = F.normalize(mlp_brain_projection(mlp_latent), dim=-1)
    mlp_text = F.normalize(mlp_text_projection(text_embeddings.to(device)), dim=-1)
    mlp_similarity = (mlp_brain @ mlp_text.T).cpu()

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, matrix, title in zip(axes, (mlp_similarity, cnn_similarity), ("MLP", "Atlas-free CNN")):
    image = ax.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(examples)), descriptions, rotation=25, ha="right")
    ax.set_yticks(range(len(examples)), [item[0] for item in examples])
    ax.set_title(title)
    for i in range(len(examples)):
        for j in range(len(examples)):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=axes, label="cosine similarity", shrink=0.8)

## Measure autoencoder inference time

This is a small illustrative benchmark, not a throughput study. It uses the same two maps and reports median milliseconds per map after warm-up.

In [ ]:
def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()

def median_ms_per_map(model, inputs, repeats=20):
    inputs = inputs.to(device)
    with torch.inference_mode():
        for _ in range(3):
            model(inputs)
        synchronize()
        timings = []
        for _ in range(repeats):
            start = time.perf_counter()
            model(inputs)
            synchronize()
            timings.append((time.perf_counter() - start) * 1000 / len(inputs))
    return float(np.median(timings))

timings = {
    "MLP": median_ms_per_map(mlp_autoencoder, mlp_inputs),
    "Atlas-free CNN": median_ms_per_map(cnn_autoencoder, cnn_inputs),
}
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(timings.keys(), timings.values(), color=["#2a78d6", "#008300"])
ax.bar_label(bars, fmt="%.2f ms")
ax.set_ylabel("Median milliseconds per map")
ax.set_title(f"Autoencoder inference on {device.type.upper()}")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
timings

## Next steps

For text-to-brain generation, load a domain branch such as `load_model("text_to_brain_cnn_pubmed")` and pass it a normalized, empty-centered SPECTER2 embedding. Baseline branches that retain the mixed autoencoder use names such as `text_to_brain_cnn_mixed_to_pubmed`.